In [1]:
import numpy as np
import matplotlib.pyplot as plt
from batchdetect.mixture import HeavyMixture,parametric_bootstrap_lrt
from typing import Callable, Any, Dict
from tqdm import  trange

In [2]:
def bootstrap_lrt_mixture(
    X,  
    null_model_factory: Callable[[int], Any],
    alt_model_factory: Callable[[int], Any],
    n_bootstrap: int = 500,
    master_seed: int | None = None,
) -> Dict[str, Any]:
    X = np.asarray(X)
    if X.ndim == 1:
        X = X[:, None]
    n_samples = X.shape[0]

    rng = np.random.default_rng(master_seed)

    # Fit null and alternative models on the observed data
    null_seed = int(rng.integers(0, 2**32 - 1))
    alt_seed = int(rng.integers(0, 2**32 - 1))

    null_model = null_model_factory(null_seed)
    null_model.fit(X)

    alt_model = alt_model_factory(alt_seed)
    alt_model.fit(X)

    # Compute observed log-likelihoods and LR statistic
    ll_null = float(null_model.score(X)) * n_samples
    ll_alt = float(alt_model.score(X)) * n_samples
    lr_obs = 2.0 * (ll_alt - ll_null)

    lr_bootstrap = np.empty(n_bootstrap, dtype=float)

    for b in trange(n_bootstrap):
        # Sample from fitted null
        Xb = null_model.sample(n_samples)
        if isinstance(Xb, tuple):
            Xb = Xb[0]
        Xb = np.asarray(Xb)
        if Xb.ndim == 1:
            Xb = Xb[:, None]

        # New seeds for each bootstrap fit
        null_seed_b = int(rng.integers(0, 2**32 - 1))
        alt_seed_b = int(rng.integers(0, 2**32 - 1))

        null_b = null_model_factory(null_seed_b)
        alt_b = alt_model_factory(alt_seed_b)

        null_b.fit(Xb)
        alt_b.fit(Xb)

        ll_null_b = float(null_b.score(Xb)) * n_samples
        ll_alt_b = float(alt_b.score(Xb)) * n_samples
        lr_bootstrap[b] = 2.0 * (ll_alt_b - ll_null_b)

    # Empirical p-value (one-sided, large LR means more evidence)
    p_value = (1.0 + np.sum(lr_bootstrap >= lr_obs)) / (n_bootstrap + 1.0)

    frac_negative = float(np.mean(lr_bootstrap < 0.0))

    return {
        "statistic": lr_obs,
        "p_value": p_value,
        "frac_negative_lr": frac_negative,
    }

def null_factory(seed: int) -> HeavyMixture:
    # K = 1 component
    return HeavyMixture(
        n_components=1,
        component_distribution='gennorm',
        n_init=3,
        max_iter=1000,
    )

def alt_factory(seed: int) -> HeavyMixture:
    # K = 2 components
    return HeavyMixture(
        n_components=2,
        component_distribution='gennorm',
        n_init=3,
        max_iter=1000,
    )

def combined_test(data):
    n = len(data)
    result = bootstrap_lrt_mixture(
    data,
    null_model_factory=null_factory,
    alt_model_factory=alt_factory,
    n_bootstrap=1000,        # increase to 1000+ for serious use
    master_seed=42,
    )
    return result

In [3]:
import pickle
with open('Batch_0.p','rb') as f:
    myDict = pickle.load(f)

In [4]:
logZs = np.array(myDict['logZ'])

In [5]:
test_results = combined_test(logZs)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:11<00:00, 89.97it/s]


In [6]:
import os

# Number of batch files
n_batches = 200

# Preallocate array for p-values
p_values = np.zeros(n_batches)

test_results_array = []

save_dict = {}


for i in range(n_batches):
    print(i)
    fname = f"Batch_{i}.p"
    if not os.path.exists(fname):
        raise FileNotFoundError(f"File not found: {fname}")
    
    with open(fname, "rb") as f:
        myDict = pickle.load(f)
    
    logZs = np.array(myDict["logZ"])
    save_dict['batch_%d'%int(i)] = logZs
    test_results = combined_test(logZs)
    test_results_array.append(test_results)
    p_values[i] = test_results["p_value"]

0


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:10<00:00, 91.53it/s]


1


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:10<00:00, 94.79it/s]


2


...

46


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:11<00:00, 90.86it/s]


47


 48%|███████████████████████████████████████████████████████                                                             | 475/1000 [00:05<00:06, 82.95it/s]


KeyboardInterrupt: 

In [7]:

save_dict = {}


for i in range(n_batches):
    print(i)
    fname = f"Batch_{i}.p"
    if not os.path.exists(fname):
        raise FileNotFoundError(f"File not found: {fname}")
    
    with open(fname, "rb") as f:
        myDict = pickle.load(f)
    
    logZs = np.array(myDict["logZ"])
    save_dict['batch_%d'%int(i)] = logZs

with open('LogLikelihoods.p','wb') as f:
    pickle.dump(save_dict,f)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199


In [ ]:
import numpy as np

def fit_beta1b_conjugate(p_values, alpha0=1.0, lambda0=1.0):
    """
    Conjugate Bayesian update for b in Beta(1, b) model.
    Prior: b ~ Gamma(alpha0, lambda0) (shape, rate).
    Returns posterior shape and rate, plus mean and MAP.
    """
    p_values = np.asarray(p_values)
    n = p_values.size
    # Sufficient statistic
    S = np.sum(np.log(1.0 - p_values))
    # Posterior hyperparameters
    alpha_post = alpha0 + n
    lambda_post = lambda0 - S   # S <= 0, so lambda_post >= lambda0
    # Posterior mean and MAP (for alpha_post > 1)
    post_mean = alpha_post / lambda_post
    if alpha_post > 1:
        post_map = (alpha_post - 1.0) / lambda_post
    else:
        post_map = None
    return {
        "alpha_post": alpha_post,
        "lambda_post": lambda_post,
        "post_mean": post_mean,
        "post_map": post_map
    }

# Example usage with your 176 p-values:
results = fit_beta1b_conjugate(p_values, alpha0=1.0, lambda0=1.0)
print(results)


In [ ]:
from scipy.stats import gamma

lower = gamma.ppf(0.025, a=results['alpha_post'], scale=1.0/results['lambda_post'])
upper = gamma.ppf(0.975, a=results['alpha_post'], scale=1.0/results['lambda_post'])
print(lower,upper)

In [ ]:
myDict2 = {'p_values':p_values,'results':results}
with open('GenNorm.p','wb') as f:
    pickle.dump(myDict2,f)

array([0.42557443, 0.64535465, 0.45454545, 0.16083916, 0.08891109,
       0.73326673, 0.73026973, 0.998002  , 0.24475524, 0.56743257,
       0.40659341, 0.05594406, 0.37662338, 0.39060939, 0.12687313,
       0.88711289, 0.32967033, 0.43256743, 0.13386613, 0.25174825,
       0.11788212, 0.40959041, 0.82317682, 0.29270729, 0.26973027,
       0.38561439, 0.10989011, 0.30569431, 0.2027972 , 0.13186813,
       0.92007992, 0.44555445, 0.1958042 , 0.44355644, 0.22277722,
       0.51348651, 0.33566434, 0.32867133, 0.46353646, 0.27672328,
       0.4955045 , 0.86013986, 0.24775225, 0.03996004, 0.93006993,
       0.07392607, 0.26573427, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.     